# 05 — Signal Backtest
Rule-based signal backtest, threshold sensitivity, ML predictor layer.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# signals.py API:
#   detect_shocks(returns, threshold, window, col, cooldown) -> List[Signal]
#   backtest_signals(signals, returns, tc) -> List[Signal]
#   performance_metrics(signals) -> Dict
#   signals_to_df(signals) -> pd.DataFrame
from signals import detect_shocks, backtest_signals, performance_metrics, signals_to_df
from utils import DATA_PROC, TABLES_DIR, PLOTS_DIR, set_theme, save_table
from constants import MARKET_COL, OIL_COL, SECTORS


In [2]:
returns = pd.read_parquet(DATA_PROC / 'returns.parquet')
print(f'Returns: {returns.shape}')
print(f'Date range: {returns.index.min().date()} — {returns.index.max().date()}')
print(f'Columns: {list(returns.columns)}')


Returns: (1928, 18)
Date range: 2019-01-02 — 2026-05-25
Columns: ['RELIANCE', 'ONGC', 'IOC', 'BPCL', 'INDIGO', 'HPCL', 'ADANIPORTS', 'TATAMOTORS', 'MARUTI', 'ASIANPAINT', 'HINDUNILVR', 'ITC', 'TCS', 'INFY', 'CIPLA', 'BRENT', 'NIFTY', 'USDINR']


In [3]:
# Detect Brent shocks (>3% over any 2-day window, 10-day cooldown)
raw_signals = detect_shocks(returns, threshold=0.03, window=2, cooldown=10)
print(f'Total signals detected : {len(raw_signals)}')
print(f'  Up-shock   : {sum(1 for s in raw_signals if s.direction == "up")}')
print(f'  Down-shock : {sum(1 for s in raw_signals if s.direction == "down")}')
print('\nFirst 3 signals:')
for s in raw_signals[:3]: print(' ', s)


Total signals detected : 149
  Up-shock   : 75
  Down-shock : 74

First 3 signals:
  Signal(trigger_date=Timestamp('2019-01-03 00:00:00'), entry_date=Timestamp('2019-01-04 00:00:00'), exit_date=Timestamp('2019-01-09 00:00:00'), brent_move=-4.18, direction='down', longs=['AUTO', 'FMCG'], shorts=['OIL_GAS'], pnl=0.0, pnl_long=0.0, pnl_short=0.0, hit=False, details={})
  Signal(trigger_date=Timestamp('2019-01-22 00:00:00'), entry_date=Timestamp('2019-01-23 00:00:00'), exit_date=Timestamp('2019-01-28 00:00:00'), brent_move=3.18, direction='up', longs=['OIL_GAS'], shorts=['AUTO', 'FMCG'], pnl=0.0, pnl_long=0.0, pnl_short=0.0, hit=False, details={})
  Signal(trigger_date=Timestamp('2019-02-06 00:00:00'), entry_date=Timestamp('2019-02-07 00:00:00'), exit_date=Timestamp('2019-02-12 00:00:00'), brent_move=3.06, direction='up', longs=['OIL_GAS'], shorts=['AUTO', 'FMCG'], pnl=0.0, pnl_long=0.0, pnl_short=0.0, hit=False, details={})


In [4]:
# Run full backtest (entry Day+1, exit Day+3, 10bps round-trip costs)
filled_signals = backtest_signals(raw_signals, returns, tc=0.0010)
sig_df = signals_to_df(filled_signals)
print(f'Filled signals: {len(filled_signals)}')
print()
print(sig_df[['trigger_date','direction','brent_move','pnl_pct','pnl_long','pnl_short','hit']].to_string(index=False))


Filled signals: 149

trigger_date direction  brent_move  pnl_pct  pnl_long  pnl_short   hit
  2019-01-03      down       -4.18   0.1956    1.4638    -0.8726  True
  2019-01-22        up        3.18   0.2469   -0.1688     0.8626  True
  2019-02-06        up        3.06  -0.8735   -1.8289     0.2818 False
  2019-02-20        up        4.30  -1.2091   -1.5355    -0.6828 False
  2019-03-06      down       -5.73  -0.5362   -0.5555    -0.3169 False
  2019-03-21      down       -3.76  -0.8740    0.9598    -2.5078 False
  2019-04-09      down       -3.74  -0.3624   -0.2950    -0.2299 False
  2019-04-26      down       -3.25   1.8682    3.1751     0.7613  True
  2019-05-22      down       -3.06   1.4203    2.8230     0.2176  True
  2019-06-05      down       -3.91  -0.3675   -0.3026    -0.2325 False
  2019-06-19      down       -3.41   0.1761    1.2679    -0.7157  True
  2019-07-08      down       -3.28  -0.1456    0.6334    -0.7246 False
  2019-07-29      down       -3.61   0.1428    0.5499   

In [5]:
# Performance summary
metrics = performance_metrics(filled_signals)
print('='*50)
print('FULL STRATEGY PERFORMANCE')
print('='*50)
for k, v in metrics.items():
    print(f'  {k:<22} {v}')

# Split by direction
up_sigs   = [s for s in filled_signals if s.direction == 'up']
down_sigs = [s for s in filled_signals if s.direction == 'down']

if up_sigs:
    up_m = performance_metrics(up_sigs)
    print(f"\nUp-shock only   → hit: {up_m['hit_ratio']:.0%}  sharpe: {up_m['sharpe_trade']}  mean pnl: {up_m['mean_pnl_pct']}%")
if down_sigs:
    dn_m = performance_metrics(down_sigs)
    print(f"Down-shock only → hit: {dn_m['hit_ratio']:.0%}  sharpe: {dn_m['sharpe_trade']}  mean pnl: {dn_m['mean_pnl_pct']}%")

save_table(pd.DataFrame([metrics]), 'strategy_metrics')
save_table(sig_df, 'trade_log')


FULL STRATEGY PERFORMANCE
  n_trades               149
  hit_ratio              0.4832
  mean_pnl_pct           -0.112
  std_pnl_pct            1.033
  sharpe_trade           -0.108
  sharpe_ann             -0.487
  max_drawdown           -22.72
  total_return           -16.05
  cagr                   -2.34
  best_trade             2.201
  worst_trade            -3.37
  profit_factor          0.761

Up-shock only   → hit: 53%  sharpe: -0.079  mean pnl: -0.077%
Down-shock only → hit: 43%  sharpe: -0.135  mean pnl: -0.147%
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\strategy_metrics.csv
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\trade_log.csv


,trigger_date,entry_date,exit_date,brent_move,direction,longs,shorts,pnl_pct,pnl_long,pnl_short,hit
0,2019-01-03,2019-01-04,2019-01-09,-4.18,down,AUTO+FMCG,OIL_GAS,0.1956,1.4638,-0.8726,True
1,2019-01-22,2019-01-23,2019-01-28,3.18,up,OIL_GAS,AUTO-FMCG,0.2469,-0.1688,0.8626,True
2,2019-02-06,2019-02-07,2019-02-12,3.06,up,OIL_GAS,AUTO-FMCG,-0.8735,-1.8289,0.2818,False
3,2019-02-20,2019-02-21,2019-02-26,4.30,up,OIL_GAS,AUTO-FMCG,-1.2091,-1.5355,-0.6828,False
4,2019-03-06,2019-03-07,2019-03-12,-5.73,down,AUTO+FMCG,OIL_GAS,-0.5362,-0.5555,-0.3169,False
...,...,...,...,...,...,...,...,...,...,...,...
144,2026-03-18,2026-03-19,2026-03-24,6.91,up,OIL_GAS,AUTO-FMCG,0.0490,-2.5180,2.8160,True
145,2026-04-01,2026-04-02,2026-04-07,-10.87,down,AUTO+FMCG,OIL_GAS,1.5091,1.6926,1.5256,True
146,2026-04-15,2026-04-16,2026-04-21,-6.57,down,AUTO+FMCG,OIL_GAS,0.9330,2.6566,-0.5905,True
147,2026-04-29,2026-04-30,2026-05-05,8.67,up,OIL_GAS,AUTO-FMCG,-0.5825,-0.8906,-0.0744,False


In [6]:
# Equity curve + drawdown
set_theme()
pnls    = np.array([s.pnl / 100 for s in filled_signals])
equity  = np.cumprod(1 + pnls)
running_max = np.maximum.accumulate(equity)
drawdown    = (equity - running_max) / running_max

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax1.plot(range(len(equity)), equity, color='#58A6FF', lw=2.2, label='Oil Shock Strategy')
ax1.fill_between(range(len(equity)), 1, equity, alpha=0.08, color='#58A6FF')
ax1.axhline(1, color='#555', lw=0.7)
for i, s in enumerate(filled_signals):
    ax1.scatter(i, equity[i], color='#3FB950' if s.hit else '#F85149', s=28, zorder=5)
ax1.set_ylabel('Portfolio Value (₹1 start)')
ax1.set_title('Oil Shock Strategy — Equity Curve & Drawdown')
ax1.legend(fontsize=9); ax1.grid(True)

ax2.fill_between(range(len(drawdown)), drawdown * 100, 0, alpha=0.7, color='#F85149')
ax2.set_ylabel('Drawdown %'); ax2.set_xlabel('Trade #')
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
ax2.grid(True)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Max drawdown: {drawdown.min()*100:.2f}%')


Max drawdown: -22.72%


In [7]:
# PnL distribution
set_theme()
sorted_pnls = sorted(pnls * 100)
colors_bar  = ['#3FB950' if p > 0 else '#F85149' for p in sorted_pnls]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(len(sorted_pnls)), sorted_pnls, color=colors_bar, edgecolor='#0D1117', lw=0.5)
ax1.axhline(0, color='#8B949E', lw=0.8)
ax1.set_xlabel('Trade (sorted by PnL)'); ax1.set_ylabel('PnL %')
ax1.set_title('Trade PnL Distribution'); ax1.grid(True)

if len(pnls) >= 5:
    roll = pd.Series(pnls).rolling(5).apply(lambda x: x.mean()/x.std() if x.std()>0 else 0)
    ax2.plot(roll.values, color='#F0A830', lw=2)
    ax2.axhline(0, color='#8B949E', lw=0.7)
    ax2.axhline(1, color='#3FB950', lw=0.7, ls='--', alpha=0.6, label='Sharpe=1')
    ax2.set_xlabel('Trade #'); ax2.set_ylabel('Rolling Sharpe (5-trade)')
    ax2.set_title('Rolling Strategy Quality'); ax2.legend(fontsize=9); ax2.grid(True)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'pnl_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [8]:
# Threshold sensitivity
thresholds = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06]
sens_rows  = []
for thr in thresholds:
    sigs   = detect_shocks(returns, threshold=thr, window=2, cooldown=10)
    filled = backtest_signals(sigs, returns)
    if len(filled) < 3:
        continue
    m = performance_metrics(filled)
    sens_rows.append({'threshold_%': thr*100, 'n_trades': m['n_trades'],
                      'hit_ratio': m['hit_ratio'], 'sharpe_trade': m['sharpe_trade'],
                      'mean_pnl': m['mean_pnl_pct'], 'max_dd': m['max_drawdown']})

sens_df = pd.DataFrame(sens_rows)
print(sens_df.to_string(index=False))
save_table(sens_df, 'threshold_sensitivity')

set_theme()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(sens_df['threshold_%'], sens_df['sharpe_trade'], color='#58A6FF', marker='o', lw=2)
ax1.axvline(3.0, color='#F85149', ls='--', lw=1, alpha=0.7, label='Base (3%)')
ax1.set_xlabel('Shock threshold (%)'); ax1.set_ylabel('Trade-level Sharpe')
ax1.set_title('Sharpe vs Threshold'); ax1.legend(fontsize=9); ax1.grid(True)

ax2.plot(sens_df['threshold_%'], sens_df['n_trades'], color='#F0A830', marker='s', lw=2)
ax2.set_xlabel('Shock threshold (%)'); ax2.set_ylabel('Number of trades')
ax2.set_title('Trade Count vs Threshold'); ax2.grid(True)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()


 threshold_%  n_trades  hit_ratio  sharpe_trade  mean_pnl  max_dd
         1.0       186     0.4516        -0.082    -0.076 -20.076
         2.0       169     0.4734        -0.047    -0.044 -17.217
         3.0       149     0.4832        -0.108    -0.112 -22.720
         4.0       119     0.5378         0.011     0.011 -11.617
         5.0        86     0.5581         0.034     0.035  -5.358
         6.0        58     0.5345         0.067     0.072  -7.093
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\threshold_sensitivity.csv


In [9]:
# ML predictor layer
# train_all_sectors(returns, event_dates, sectors, target_hold) -> Dict[str, SectorPredictor]
# Sector names are resolved via SECTOR_MAP in constants.py
from predictive import train_all_sectors

event_dates = [s.trigger_date for s in filled_signals]
print(f'Training {len(SECTORS)} sector predictors on {len(event_dates)} event dates ...')
predictors = train_all_sectors(returns, event_dates, SECTORS, target_hold=3)
print(f'Trained: {list(predictors.keys())}')


Training 5 sector predictors on 149 event dates ...
  ✓  OIL_GAS: LOO R²=-0.1662  top=brent_5d_ret
  ✓  AUTO: LOO R²=-0.1669  top=brent_5d_ret
  ✓  FMCG: LOO R²=0.1686  top=brent_5d_ret
  ✓  IT: LOO R²=-0.1191  top=brent_vol_20d
  ✓  PHARMA: LOO R²=-0.1755  top=nifty_vol_20d
Trained: ['OIL_GAS', 'AUTO', 'FMCG', 'IT', 'PHARMA']


In [10]:
# Feature importance table
rows = []
for sec, pred in predictors.items():
    s   = pred.summary()
    row = {'sector': sec, 'loo_ic': s['loo_r2'], 'top_feature': s['top_feature']}
    row.update({k: round(v, 4) for k, v in s['coefs'].items()})
    rows.append(row)

fi_df = pd.DataFrame(rows)
print(fi_df[['sector', 'loo_ic', 'top_feature']].to_string(index=False))
save_table(fi_df, 'ml_feature_importance')


 sector  loo_ic   top_feature
OIL_GAS -0.1662  brent_5d_ret
   AUTO -0.1669  brent_5d_ret
   FMCG  0.1686  brent_5d_ret
     IT -0.1191 brent_vol_20d
 PHARMA -0.1755 nifty_vol_20d
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\ml_feature_importance.csv


,sector,loo_ic,top_feature,brent_shock,brent_5d_ret,brent_vol_20d,nifty_vol_20d,sector_mom_5d,brent_sector_corr_60d
0,OIL_GAS,-0.1662,brent_5d_ret,-0.0001,-0.0009,-0.0001,0.0001,0.0001,0.0000
1,AUTO,-0.1669,brent_5d_ret,-0.0004,-0.0012,0.0006,0.0000,0.0001,0.0007
2,FMCG,0.1686,brent_5d_ret,-0.0009,-0.0014,-0.0004,-0.0003,0.0013,-0.0002
3,IT,-0.1191,brent_vol_20d,0.0004,-0.0010,0.0013,-0.0001,0.0013,-0.0004
4,PHARMA,-0.1755,nifty_vol_20d,-0.0003,-0.0003,-0.0006,-0.0006,-0.0004,-0.0000


In [11]:
# ML-filtered backtest: only take signals where ML confirms direction
from constants import SECTOR_MAP

ml_filtered = []
for s in filled_signals:
    pred = predictors.get('OIL_GAS')
    if pred is None:
        ml_filtered.append(s)
        continue
    idx = returns.index.searchsorted(s.trigger_date)
    if idx < 60:
        continue
    brent      = returns[OIL_COL]
    nifty      = returns[MARKET_COL]
    oil_stocks = [c for c in SECTOR_MAP['OIL_GAS'] if c in returns.columns]
    sec_series = returns[oil_stocks].mean(axis=1)

    feat = {
        'brent_shock':           float(brent.iloc[idx]),
        'brent_5d_ret':          float(brent.iloc[idx-5:idx].sum()),
        'brent_vol_20d':         float(brent.iloc[idx-20:idx].std() * (252**0.5)),
        'nifty_vol_20d':         float(nifty.iloc[idx-20:idx].std() * (252**0.5)),
        'sector_mom_5d':         float(sec_series.iloc[idx-5:idx].sum()),
        'brent_sector_corr_60d': float(brent.iloc[idx-60:idx].corr(sec_series.iloc[idx-60:idx])),
    }
    pred_ret = pred.predict_single(feat)
    if (s.direction == 'up' and pred_ret > 0) or (s.direction == 'down' and pred_ret < 0):
        ml_filtered.append(s)

print(f'ML-filtered signals: {len(ml_filtered)} of {len(filled_signals)}')
if ml_filtered:
    ml_m   = performance_metrics(ml_filtered)
    base_m = performance_metrics(filled_signals)
    print(f"\nML-filtered — hit: {ml_m['hit_ratio']:.0%}  sharpe: {ml_m['sharpe_trade']}  mean pnl: {ml_m['mean_pnl_pct']}%")
    print(f"Base         — hit: {base_m['hit_ratio']:.0%}  sharpe: {base_m['sharpe_trade']}  mean pnl: {base_m['mean_pnl_pct']}%")
print('NB05 complete ✓')


ML-filtered signals: 44 of 149

ML-filtered — hit: 36%  sharpe: -0.368  mean pnl: -0.415%
Base         — hit: 48%  sharpe: -0.108  mean pnl: -0.112%
NB05 complete ✓
